In [4]:
import os
import time
import re
import numpy as np
import pandas as pd
import matplotlib as mpl
import gzip

In [5]:
# Dealing with VCF header
class VCFMetaEntry(dict):
    """
    Class for processing VCF metadata entries so they can be used for parsing the main file contents
    """
    
    meta_id = None
    number = None
    desc = None
    type = None
    
    
    def __init__(self, field):
        self.field = field


def readheader(vcf_path):
    header_fields = []
    nheader = 0
    with open(vcf_path, 'r') as vcf_file:
        for line in vcf_file:
            if line.startswith("##"):
                header_fields.append(line)
                nheader += 1
            else:
                break


def readheaderline(line):
    """
    Reads an unprocessed line from VCF header and returns a VCFHeaderEntry object
    """

    try:
        entry = VCFMetaEntry(re.search(r'##[A-Z]+=', line).group().replace("##", "").replace("=", ""))
        entry.meta_id = re.search(r'ID=([A-Z]|[a-z]|[0-9]|_)+', line).group().replace("ID=", "")
    except AttributeError:
        return None

    try:
        number_val = int(re.search(r'Number=([0-9]|\.)+,', line).group().replace("Number=", "").replace(",", ""))
        entry.number = number_val
    except ValueError:
        number_val = re.search(r'Number=([0-9]|\.)+,', line).group().replace("Number=", "").replace(",", "")
        entry.number = number_val
    except AttributeError:
        entry.number = None

    try:
        type_val = re.search(r'Type=([A-Z]|[a-z]|[0-9])+,', line).group().replace("Type=", "").replace(",", "")
        entry.type = type_val
    except AttributeError:
        entry.type = None

    try:
        entry.desc = re.search(r'Description=".+"', line).group().replace('Description=\"', '').replace('"', '')
    except AttributeError:
        entry.desc = None
    
    return entry
        
    

In [6]:
# Global vars
wd = '/stornext/Bioinf/data/bioinf-data/Papenfuss_lab_projects/SVensemble2/vcf_parsing'
gridss_path = f'{wd}/data/test_gridss_output.NA12878.vcf'

gnomad_path = '/home/users/allstaff/shikov.v/projects/SVensemble2/data/gnomad/4.1.0/downloads/gnomad.v4.1.sv.sites.vcf.gz'

vcf_path = gnomad_path

In [7]:
def get_read_params(file_path):
    """
    Checks the supplied path for file extensions and returns a tibble containing:
    - the module from which .open function will be called (may change it later)
    - the read mode string for that .open function
    """
    is_gzip = re.search(r'\.(gz|gzip)$', file_path)

    if is_gzip:
        vcf_file_handler_mod = gzip
        readmode = 'rt'
    else:
        vcf_file_handler_mod = __builtins__
        readmode = 'r'

    return vcf_file_handler_mod, readmode

In [8]:
def parse_vcf(
    vcf_path, 
    filter_delim=';', 
    info_delim=';', info_eq='=', 
    format_present='infer', format_delim=':',
    line_limit=0, report_freq=100000
):
    """
    
    """
    
    header_dict = {
        'FILTER': {},
        'INFO': {},
        'FORMAT': {}
    }
    
    vcf_data = {
        'CHROM': [],
        'POS': [],
        'ID': [],
        'REF': [],
        'ALT': [],
        'QUAL': [],
        'INFO': []
    }

    if format_present == 'infer':
        vcf_file_handler_mod, readmode = get_read_params(vcf_path)
    
    header_nlines = 0
    with vcf_file_handler_mod.open(vcf_path, readmode) as vcf:
        
        start_time = time.time()
        for i, line in enumerate(vcf):
            if i % report_freq == 0:
                current_time = time.time()
                print(f'Processed {i} entries out of {line_limit}. Elapsed time: {current_time - start_time} seconds.')
    
            
            # There are 2 major blocks processed differently: the header and the body
            # Currently the algorithm to process a headerless vcf file is not done, so a header is mandatory
    
            if line.startswith('#'):
                header_nlines += 1
                # Both the metadata and the header itself has a # / ## in the beginning
                if line.startswith('##'):
                    # print(line)
                    header_entry = readheaderline(line)
                    if header_entry != None:
                        if header_entry.field in header_dict.keys():
                            header_dict[header_entry.field].update({f'{header_entry.field}_{header_entry.meta_id}': (header_entry.number, header_entry.type, header_entry.desc)})
                            # print(f'[{header_entry.field}] {header_entry.meta_id}, number: {header_entry.number}, type: {header_entry.type}, desc: {header_entry.desc}')
                            vcf_data.update({f'{header_entry.field}_{header_entry.meta_id}': []})
                else:
                    continue        
            else:
                if (i - header_nlines >= line_limit) & (line_limit != 0):
                    # print(vcf_data)
                    # print(header_dict)
                    break
                
                line_list = line.split('\t')
                # print(line_list)
    
                # First fields are easily processed
                vcf_data['CHROM'].append(line_list.pop(0))
                vcf_data['POS'].append(line_list.pop(0))
                vcf_data['ID'].append(line_list.pop(0))
                vcf_data['REF'].append(line_list.pop(0))
                vcf_data['ALT'].append(line_list.pop(0))
                vcf_data['QUAL'].append(line_list.pop(0))
    
                # FILTER
                # The filter field is referenced to all possible filter field in the metadata
                # If an entry is present, then the column is saved with value 1, else - with 0
                filter_entry_with_prefix = [f'FILTER_{x}' for x in line_list.pop(0).split(filter_delim)]
                # print(filter_entry_with_prefix)
                
                for filter_field in header_dict['FILTER'].keys():
                    if filter_field in filter_entry_with_prefix:
                        vcf_data[filter_field].append(1)
                    else:
                        vcf_data[filter_field].append(0)
    
                # INFO
                # Here the program iterates over the possible INFO field values listed in the header too
                # info_entry_with_prefix_raw = [f'INFO_{x}' for x in line_list.pop(0).split(info_delim)]
                info_dict = {}
                for info_item_raw in line_list.pop(0).split(info_delim):
                    info_item = info_item_raw.split(info_eq)
                    if len(info_item) == 1:
                        info_dict.update({f'INFO_{info_item[0]}': 1})
                    elif len(info_item) == 2:
                        info_dict.update({f'INFO_{info_item[0]}': info_item[1]})
                    else:
                        raise IndexError('INFO field entry with wrong syntaxis detected')
                        
                for info_field in header_dict['INFO'].keys():
                    if info_field in info_dict.keys():
                        vcf_data[info_field].append(info_dict[info_field])
                    else:
                        vcf_data[info_field].append(0)
    
    
                # FORMAT and genotype columns
                # Currently only a single genotype column is supported
                if line_list == []:
                    continue
                else:
                    format_dict = {
                        f'FORMAT_{x[0]}': x[1] for x in
                        zip(line_list.pop(0).split(format_delim), line_list.pop(0).split(format_delim))
                    }
        
                    
                    for format_field in header_dict['FORMAT'].keys():
                        if format_field in format_dict.keys():
                            vcf_data[format_field].append(format_dict[format_field])
    
    keys_for_deletion = []
    for vcf_data_key in vcf_data.keys():
        if len(vcf_data[vcf_data_key]) == 0:
            keys_for_deletion.append(vcf_data_key)
    for key_for_deletion in keys_for_deletion:
        del vcf_data[key_for_deletion]

    vcf_data = pd.DataFrame(vcf_data)

    vcf_data.POS = pd.to_numeric(vcf_data.POS)
    vcf_data.QUAL = pd.to_numeric(vcf_data.QUAL, errors='coerce')
    
    current_time = time.time()
    print(f'Finished processing. Processed {i - header_nlines} entries out of {len(vcf_data)}. Elapsed time: {current_time - start_time} seconds.')
    
    return vcf_data

In [24]:
gnomad_df = parse_vcf(gnomad_path, line_limit=100000)

Processed 0 entries out of 100000. Elapsed time: 0.0003859996795654297 seconds.
Processed 100000 entries out of 100000. Elapsed time: 37.55777406692505 seconds.
Finished processing. Processed 100000 entries out of 100000. Elapsed time: 45.40520143508911 seconds.


In [32]:
gridss_df = parse_vcf(gridss_path, line_limit=100000)

Processed 0 entries out of 100000. Elapsed time: 0.0001010894775390625 seconds.
Processed 100000 entries out of 100000. Elapsed time: 5.7131876945495605 seconds.
Finished processing. Processed 100000 entries out of 100000. Elapsed time: 7.412529945373535 seconds.


In [33]:
gridss_df = parse_vcf_experimental(gridss_path, line_limit=100000)

Processed 0 entries out of 100000. Elapsed time: 4.839897155761719e-05 seconds.
Processed 100000 entries out of 100000. Elapsed time: 5.763815402984619 seconds.
Finished processing. Processed 100000 entries out of 100000. Elapsed time: 7.456828594207764 seconds.


In [34]:
(gridss_df.POS > 10500) & (gridss_df.POS < 10600)

0        False
1        False
2        False
3        False
4        False
         ...  
99995    False
99996    False
99997    False
99998    False
99999    False
Name: POS, Length: 100000, dtype: bool

In [35]:
pd.unique(gnomad_df.CHROM)

array(['chr1'], dtype=object)

In [25]:
gnomad_df = {chrom: gnomad_df.loc[gnomad_df.CHROM == chrom] for chrom in pd.unique(gridss_df.CHROM)}

In [29]:
report_freq = 10000
truth_list = []
i = 0
pos_series = gridss_df.POS
chrom_series = gridss_df.CHROM

if set(gnomad_df.keys()) != set(chrom_series.unique()):
    raise KeyError('Chromosomes in prediction dataset do not match chromosomes in gnomad')

for pos, chrom in zip(pos_series, chrom_series):
    # print(f'{chrom}: {pos}')
    # overlapping_ref_positions = (gnomad_df[chrom].POS > pos - 25) & (gnomad_df[chrom].POS < pos + 25)

    truth_list.append(
        ((gnomad_df[chrom].POS > pos - 25) & (gnomad_df[chrom].POS < pos + 25)).any()
    )
    
    i+=1
    if i % report_freq == 0:
        print(f'processed {i} entries')
    


processed 10000 entries
processed 20000 entries
processed 30000 entries
processed 40000 entries
processed 50000 entries
processed 60000 entries
processed 70000 entries
processed 80000 entries
processed 90000 entries
processed 100000 entries


In [ ]:
truth_list